In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

print("Librerías importadas correctamente.")

# Demanda y capacidad superior a 2MW (Basado en Transformadores Tr2)

In [ ]:
# Definir la ruta del archivo excel fuente
file_name = "SE LAT Circular CREG 014-2022 Anexo2_TS_ND Octubre 2025.xlsx"
excel_file = "/home/oscar/Documents/WePower/" + file_name

if not os.path.exists(excel_file):
    excel_file = file_name

print(f"Importando excel desde: {excel_file}")
xl = pd.ExcelFile(excel_file)

In [ ]:
# Cargar hojas
dfs = {sheet_name: pd.read_excel(excel_file, sheet_name=sheet_name, header=3) for sheet_name in xl.sheet_names}
print("Hojas cargadas correctamente.")

In [ ]:
# 1. Procesar Tr2
df_tr2 = dfs.get('3. Tr2').copy()
tr2_data = df_tr2[['Subestación', 'Capacidad [MVA]', 'Nivel de tensión HV [kV]', 'Nivel de tensión LV [kV]']].dropna(subset=['Subestación', 'Capacidad [MVA]'])
tr2_data['Capacidad_MVA'] = pd.to_numeric(tr2_data['Capacidad [MVA]'], errors='coerce')
tr2_data['HV_kV'] = pd.to_numeric(tr2_data['Nivel de tensión HV [kV]'], errors='coerce')
tr2_data['LV_kV'] = pd.to_numeric(tr2_data['Nivel de tensión LV [kV]'], errors='coerce')
tr2_filt = tr2_data[tr2_data['Capacidad_MVA'] > 2].dropna(subset=['Capacidad_MVA'])

# 2. Procesar Demanda (Agregando por subestación para evitar duplicados en el cruce)
df_demanda = dfs.get('5. Demanda').copy()
demanda_raw = df_demanda[['Subestación', 'Demanda Media']].dropna(subset=['Subestación', 'Demanda Media'])
demanda_raw['Demanda_MW'] = pd.to_numeric(demanda_raw['Demanda Media'], errors='coerce')

# Agrupamos por subestación para sumar las demandas (casos como Zipaquirá que tienen varios registros)
demanda = demanda_raw.groupby('Subestación')['Demanda_MW'].sum().reset_index()
demanda = demanda.dropna(subset=['Demanda_MW'])

# 3. Unir datos
df_merged = pd.merge(tr2_filt, demanda, on='Subestación', how='inner')

# 4. Crear el Resumen (Agrupado)
resumen = df_merged.groupby('Subestación').agg({
    'Capacidad_MVA': 'sum',
    'Demanda_MW': 'first', # Al estar pre-agregado, 'first' ya es el total de la subestación
    'HV_kV': 'max',
    'LV_kV': 'min'
}).reset_index().sort_values(by='Demanda_MW', ascending=False)

# 5. Crear el Desglose (Por transformador)
# Renombramos la columna a 'Demanda_Total_Subestacion' para aclarar que es un valor global de la SE
desglose = df_merged[['Subestación', 'Capacidad_MVA', 'HV_kV', 'LV_kV', 'Demanda_MW']].copy()
desglose = desglose.rename(columns={'Demanda_MW': 'Demanda_Total_Subestacion_MW'})
desglose = desglose.sort_values(by=['Demanda_Total_Subestacion_MW', 'Subestación'], ascending=[False, True])

print("Análisis listo. Primeras 5 subestaciones del resumen:")
resumen.head(5)

In [ ]:
# Exportar a un único archivo Excel con dos hojas
output_file = "Reporte_Capacidad_Demanda_Subestaciones.xlsx"

with pd.ExcelWriter(output_file) as writer:
    resumen.to_excel(writer, sheet_name='Resumen', index=False)
    desglose.to_excel(writer, sheet_name='Desglose', index=False)

print(f"¡Reporte exportado exitosamente a: {output_file}!")